### CineBot: A movie ticket Booking Assistant

### Tools
Tools are just methods with proper defined input and output and description


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from typing import Literal

load_dotenv()

True

In [2]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
model = ChatOpenAI(model="gpt-5-nano")

In [ ]:
class MovieShows(BaseModel):
  name : str
  timing:str

In [ ]:
struct_model = model.with_structured_output(MovieShows)
response = struct_model.invoke("Is Interstellar showing tonight at 7pm at the Downtown cinema ?")

In [5]:
response

MovieShows(name='Interstellar', timing='Unknown')

<img src="../../assets/tools_definition.png" width="800" height="300">
Tool ->  Args with type hints.

Tools are just glorified Functions/API Calls

* Description in @tool decorator parameter overrides the docstring description.
* Langchain provides a way to define docstring and tool description separately, but is not provided in @tool, it will use docstring as desciption

In [ ]:
@tool
def check_showtimes(movie_title:str) -> str:
  """Check available showtimes for a movie at the cinema.

  Args:
      movie_title: The exact title of the movie to check
  """
  fake_showtimes = {
      "interstellar": "7:00 PM and 10:15 PM",
      "dune part two": "9:30 PM only",
      "oppenheimer": "Sold out for tonight",
  }
  return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

@tool('book_seats', description = 'Book Cinema for a customer, use whenever customer wants to book/reserve a seat.')
def reserve(movie:str,seats:int) ->str:
  """Reserve Seats"""
  return f"Reserved {seats} seat for {movie}"


Arguments Schema
> Reserved argument names<br>
> Never use runtime and config as parameter of your tool, these are reserved by Langchain.
>The following parameter names are reserved and cannot be used as tool arguments. Using these names will cause runtime errors.
>Parameter name	Purpose config Reserved for passing RunnableConfig to tools internally
>runtime Reserved for ToolRuntime parameter (accessing state, context, store)
>To access runtime information, use the ToolRuntime parameter instead of naming your own arguments config or runtime.


In [13]:
class SeatBookingIput(BaseModel):
    movie_title:str = Field(description='Exact Event  Title')
    seat_count : int = Field(description='Number of seats to book', ge=1, le=10)
    preferred_row : Literal['front', 'middle', 'back'] = Field(default='middle', description='Preferred seat row')

@tool(args_schema=SeatBookingIput)
def book_seats(movie_title:str, seats:int, preferred_row:str, config:str)-> str:
    """Book Seats for a Movie"""
    return f"Booked {seats} seats for {movie_title} in row {preferred_row}"

print(book_seats.args)

{'movie_title': {'description': 'Exact Event  Title', 'title': 'Movie Title', 'type': 'string'}, 'seat_count': {'description': 'Number of seats to book', 'maximum': 10, 'minimum': 1, 'title': 'Seat Count', 'type': 'integer'}, 'preferred_row': {'default': 'middle', 'description': 'Preferred seat row', 'enum': ['front', 'middle', 'back'], 'title': 'Preferred Row', 'type': 'string'}}


### Binding VS Execution

In [15]:
model_with_tools = model.bind_tools([check_showtimes, book_seats])
response = model_with_tools.invoke("Is Interstellar showing tonight at 7pm at the Downtown cinema ?")

In [17]:
print(response.content)
print(response.tool_calls)


[{'name': 'check_showtimes', 'args': {'movie_title': 'Interstellar'}, 'id': 'call_6cuDeHl7bRO6ohfKvnvoZSMy', 'type': 'tool_call'}]


### Runtime in Tools
a runtime param in tool which our tool can use to read a lots of things in the code and otherwise as well.

In [ ]:
from langchain.tools import tool,ToolRuntime
from langchain_core.messages import HumanMessage

@tool
def get_last_movie_mentioned(movie:str, runtime:ToolRuntime) -> str:
  """Get the last movie mentioned in the chat history."""
  pass

print(get_last_movie_mentioned.args)


In [ ]:
from langgraph.store.memory import InMemoryStore
from langchain_core.tools import tool
from langchain.tools import tool,ToolRuntime
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

In [ ]:
from typing import Any


loyalty_store= InMemoryStore()


@tool
def save_favourite_genres(customer_id:str,genre:str,runtime:ToolRuntime) -> str:
  """Save a customer's facvourite movie genre for future visits"""
  runtime.store.put((customer_id,"preferences"),"favourite_genre",{"value":genre})
  return f"Got it -- I will remmeber you like {genre} movies"

@tool
def recall_favourite_genre(customer_id:str,runtime:ToolRuntime) -> str:
  """ Recall a customer's fav movie genre, if we have saved it before"""
  favourite_genre = runtime.store.get((customer_id,"preferences"),"favourite_genre")
  return favourite_genre.value["value"] if favourite_genre else "We don't have any saved preference for this user"


memory_agent = create_agent(
    model = model,
    tools=[save_favourite_genres,recall_favourite_genre],
    store=loyalty_store  # Attached to the agent, tools can access it using runtime
)



In [ ]:
result = memory_agent.invoke({"messages": [("user", "Hi, I'm customer priya_01, I love sci-fi movies, please remember that.")]})

In [ ]:
result = memory_agent.invoke({"messages": [("user", "What genre do I usually like? I'm priya_01.")]})
print(result['messages'][-1].content)

In [ ]:
items = loyalty_store.search(("priya_01", "preferences"))
for item in items:
  print(item)

In [ ]:
@tool
def log_booking_context(runtime:ToolRuntime) -> str:
  info = runtime.execution_info


#### Skippping the Model's final Polishing


In [ ]:
@tool(return_direct=True)
def get_exact_refund_policy() -> str:
    """Tell the refund policy."""
    return "Tickets are refundable up to 2 hours before showtime. No refunds after that."

direct_agent = create_agent(model="openai:gpt-5-mini", tools=[get_exact_refund_policy])
result = direct_agent.invoke({"messages": [("user", "What's your refund policy? Please explain in points")]})
print(result["messages"][-1].content)

In [ ]:
result

#### Dynamic Tool Loading & Calling

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@tool
def standard_booking(movie_title: str) -> str:
    """Book a standard seat."""
    return f"Standard seat booked for {movie_title}."

@tool
def vip_lounge_booking(movie_title: str) -> str:
    """Book a VIP lounge seat with premium service. VIP members only."""
    return f"VIP lounge seat booked for {movie_title}."



gated_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[standard_booking, vip_lounge_booking],
)

In [ ]:
result_regular = gated_agent.invoke({"messages": [("user", "Book me a VIP lounge seat for Dune?")]})

In [ ]:
result_regular

<img src="../../assets/tool_runtime_information.png" width="800" height="400">
